Resolved 69 packages in 1ms
Checked 61 packages in 1ms


In [1]:

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.registry import TASKS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Registered tasks:", len(TASKS))


Device: cuda
GPU: NVIDIA A100 80GB PCIe
Registered tasks: 38



## 1. What is one training problem?

Each sampled problem has four pieces:

- `prompt`: initial state + sequence of operations;
- `correct_trace`: canonical intermediate execution;
- `wrong_trace`: surface-matched but locally invalid execution;
- `gold`: the same terminal answer used across supervision conditions.

This is the core controlled-design idea: we can change trace supervision without changing the underlying answer problem.


In [ ]:

random.seed(7)
t = "boolean_circuit_8"
task = TASKS[t]
inst = task.sample()

print("TASK:", task.name)
print("DESCRIPTION:", task.description)
print("CHANCE ACCURACY:", task.chance_acc)
print()
print("PROMPT:")
print(inst.prompt)
print()
print("CORRECT TRACE:")
print(inst.correct_trace)
print()
print("WRONG TRACE:")
print(inst.wrong_trace)
print()
print("GOLD ANSWER:")
print(inst.gold)


if t == "state_machine_4":
    a_part, b_part, s_part, u_part = inst.prompt.split(";")

    a_table = a_part[1:]
    a_next = [a_table[i:i+2] for i in range(0, len(a_table), 2)]

    print("A transition table:")
    print("Index | Next state")
    print("------------------")

    for i, nxt in enumerate(a_next):
        print(f"{i:02d}    | {nxt}")



TASK: state_machine_4
DESCRIPTION: execute a random state machine for 4 transitions
CHANCE ACCURACY: 0.0625

PROMPT:
a03140709131104051208010015060210;b14051307091210151106040208000301;s11;uaaaa

CORRECT TRACE:
11a00 00a03 03a09 09a08

WRONG TRACE:
11a08 08a10 10a09 09a06

GOLD ANSWER:
08
A transition table:
Index | Next state
------------------
00    | 03
01    | 14
02    | 07
03    | 09
04    | 13
05    | 11
06    | 04
07    | 05
08    | 12
09    | 08
10    | 01
11    | 00
12    | 15
13    | 06
14    | 02
15    | 10


In [16]:
inst.wrong_trace.split()

['d0311', 'a0116', 'a0601', 'e0213']

In [3]:
from types import SimpleNamespace
import torch
from src.registry import TASKS
from compare_supervision import generate_unique, train_one, evaluate as evaluate_answer
from sweep_ratio import evaluate as evaluate_trace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
task = TASKS["boolean_circuit_8"]

args = SimpleNamespace(
    train_size=100_000,
    val_size=1_000,
    train_seed=501,
    val_seed=101,
    batch_seed=12345,
    batch_size=128,
    eval_batch_size=256,
    steps=8_000,
    lr=3e-4,
    weight_decay=0.0,
    grad_clip=1.0,
    embedding=128,
    heads=4,
    layers=2,
    dropout=0.0,
    workers=0,
)

/home/hariguru/aayus/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_instances = generate_unique(task, args.train_size, args.train_seed)
train_prompts = {x.prompt for x in train_instances}

val_instances = generate_unique(
    task,
    args.val_size,
    args.val_seed,
    excluded=train_prompts,
)

In [5]:
SEED = 2001

outcome_model, outcome_loss = train_one(
    task, train_instances, "outcome", SEED, args, device
)

process_model, process_loss = train_one(
    task, train_instances, "process", SEED, args, device
)

In [6]:
outcome_acc = evaluate_answer(
    outcome_model, task, val_instances, "outcome", args, device
)

process_metrics = evaluate_trace(
    process_model, task, val_instances, "process", args, device
)

print("Outcome answer accuracy:", outcome_acc)

print("Process answer accuracy:",
      process_metrics["answer_accuracy"])

print("Process exact trace accuracy:",
      process_metrics["exact_trace_accuracy"])

print("Process trace-step accuracy:",
      process_metrics["trace_step_accuracy"])

Outcome answer accuracy: 0.082
Process answer accuracy: 0.903
Process exact trace accuracy: 0.903
Process trace-step accuracy: 0.976
